In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.cluster import KMeans
import warnings

warnings.filterwarnings('ignore')

print("\n--- İLERİ SEVİYE VERİ İŞLEME (DATA PROCESSING) BAŞLIYOR ---")

# 1. Verileri Yüklüyoruz (Orijinal veriyi de katarak)
train = pd.read_csv("../data/raw/train.csv", index_col='id')
test = pd.read_csv("../data/raw/test.csv", index_col='id')

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
orig = pd.read_csv(url)
orig = orig.rename(columns={'customerID': 'id'}).set_index('id')

y_train = train['Churn'].map({'Yes': 1, 'No': 0})
train.drop('Churn', axis=1, inplace=True)

y_orig = orig['Churn'].map({'Yes': 1, 'No': 0})
orig.drop('Churn', axis=1, inplace=True)

# Eğitim için Train ve Orig birleşiyor
X_train_full = pd.concat([train, orig])
y_train_full = pd.concat([y_train, y_orig])

# Tüm veriyi Feature Engineering için birleştiriyoruz
df_all = pd.concat([X_train_full, test], keys=['train', 'test'])
df_all['TotalCharges'] = pd.to_numeric(df_all['TotalCharges'], errors='coerce').fillna(0)

# --- BİLDİĞİMİZ SİHİRLİ ÖZELLİKLER (Bunları koruyoruz, çok işe yaramıştı) ---
df_all['TotalCharges_Diff'] = df_all['TotalCharges'] - (df_all['MonthlyCharges'] * df_all['tenure'])
ek_hizmetler = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df_all['Total_Extra_Services'] = (df_all[ek_hizmetler] == 'Yes').sum(axis=1)
contract_map = {'Month-to-month': 1, 'One year': 12, 'Two year': 24}
df_all['Contract_Months'] = df_all['Contract'].map(contract_map)
df_all['Tenure_Contract_Ratio'] = df_all['tenure'] / df_all['Contract_Months']
df_all['Auto_Payment'] = df_all['PaymentMethod'].isin(['Bank transfer (automatic)', 'Credit card (automatic)']).astype(int)

# -------------------------------------------------------------------
# YENİ VERİ İŞLEME 1: GRUP İSTATİSTİKLERİ (GROUPBY FEATURES)
# -------------------------------------------------------------------
print("Grup İstatistikleri Hesaplanıyor...")
# Her bir sözleşme tipi için ortalama aylık ücreti bul ve müşterinin kendi ücretiyle farkını al
df_all['Avg_Monthly_by_Contract'] = df_all.groupby('Contract')['MonthlyCharges'].transform('mean')
df_all['Diff_From_Contract_Avg'] = df_all['MonthlyCharges'] - df_all['Avg_Monthly_by_Contract']

# İnternet servisine göre tenure (süre) ortalamasından sapma
df_all['Avg_Tenure_by_Internet'] = df_all.groupby('InternetService')['tenure'].transform('mean')
df_all['Diff_From_Tenure_Avg'] = df_all['tenure'] - df_all['Avg_Tenure_by_Internet']

# -------------------------------------------------------------------
# YENİ VERİ İŞLEME 2: K-MEANS KÜMELEME İLE SEGMENTASYON
# -------------------------------------------------------------------
print("K-Means Müşteri Segmentasyonu Yapılıyor...")
# Sadece sayısal sütunları alarak müşterileri 5 finansal segmente ayırıyoruz
cluster_features = ['tenure', 'MonthlyCharges', 'TotalCharges', 'TotalCharges_Diff']
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
# Küme numaralarını (0,1,2,3,4) modelin kategorik algılaması için string yapıyoruz
df_all['Financial_Segment'] = kmeans.fit_predict(df_all[cluster_features]).astype(str)

# One-Hot Encoding
cat_cols = df_all.select_dtypes(include=['object']).columns
df_all_encoded = pd.get_dummies(df_all, columns=cat_cols, drop_first=True)

# Veriyi tekrar Train ve Test olarak ayırıyoruz
X_train_final = df_all_encoded.xs('train')
X_test_final = df_all_encoded.xs('test')

print(f"Eğitim Seti Boyutu {X_train_final.shape[1]} Sütuna Çıktı! Modelleniyor...\n")

# --- XGBOOST EĞİTİMİ (OOF: 5-Fold) ---
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X_train_final))
test_preds = np.zeros(len(X_test_final))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train_final, y_train_full)):
    X_tr, y_tr = X_train_final.iloc[train_idx], y_train_full.iloc[train_idx]
    X_va, y_va = X_train_final.iloc[valid_idx], y_train_full.iloc[valid_idx]
    
    # En iyi skoru aldığımız parametrelerle devam ediyoruz
    model = XGBClassifier(
        n_estimators=600,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="auc"
    )
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    
    valid_preds = model.predict_proba(X_va)[:, 1]
    oof_preds[valid_idx] = valid_preds
    test_preds += model.predict_proba(X_test_final)[:, 1] / skf.n_splits
    
    print(f"Data Processing Fold {fold+1} AUC: {roc_auc_score(y_va, valid_preds):.4f}")

cv_auc = roc_auc_score(y_train_full, oof_preds)
print(f"\n🏆 İleri Seviye Veri İşleme (OOF) AUC Skoru: {cv_auc:.4f}")

# Gönderim Dosyası
submission = pd.DataFrame({'id': X_test_final.index, 'Churn': test_preds})
submission.to_csv("../submissions/submission_advanced_data_processing.csv", index=False)
print("✅ Yeni veri mühendisliği dosyası kaydedildi!")


--- İLERİ SEVİYE VERİ İŞLEME (DATA PROCESSING) BAŞLIYOR ---
Grup İstatistikleri Hesaplanıyor...
K-Means Müşteri Segmentasyonu Yapılıyor...
Eğitim Seti Boyutu 43 Sütuna Çıktı! Modelleniyor...

Data Processing Fold 1 AUC: 0.9158
Data Processing Fold 2 AUC: 0.9164
Data Processing Fold 3 AUC: 0.9140
Data Processing Fold 4 AUC: 0.9152
Data Processing Fold 5 AUC: 0.9161

🏆 İleri Seviye Veri İşleme (OOF) AUC Skoru: 0.9155
✅ Yeni veri mühendisliği dosyası kaydedildi!
